# A/B Test Significance Analysis
#1. Introduction
**Мета:** порахувати статистичну значущість чотирьох ключових конверсійних
метрик A/B-тесту через Python (замість онлайн-калькуляторів) і підготувати
CSV-файл для візуалізації в Tableau.

**Метрики, які аналізуємо:**
- add_payment_info / session
- add_shipping_info / session
- begin_checkout / session
- new_accounts / session

**Джерело даних:** той самий набір даних з BigQuery, що і в завданні
*Create Your A/B Testing Tool* — тягнемо його напряму запитом, без
ручного експорту в CSV.

**План ноутбука:**
1. Introduction
2. Environment Setup
3. Підключення до BigQuery і завантаження даних
4. Визначення метрик
5. Z-тест для двох пропорцій
6. Функція розрахунку значущості для довільного зрізу
7. Розрахунок в тоталі по кожному тесту
8. Розрахунок в розрізах
9. Збереження результату в CSV
10. Висновки та посилання

#2. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from google.cloud import bigquery
from google.colab import auth

#3. Підключення до BigQuery і завантаження даних

In [ ]:
auth.authenticate_user()

PROJECT_ID = 'data-analytics-mate'
client = bigquery.Client(project=PROJECT_ID)

query = """
WITH
  session_info AS (
    SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
    FROM `DA.ab_test` ab
    JOIN `DA.session` s
      ON ab.ga_session_id = s.ga_session_id
    JOIN `DA.session_params` sp
      ON ab.ga_session_id = sp.ga_session_id
  ),




session_with_orders AS (
SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT o.ga_session_id) AS sessions_with_orders
FROM `DA.order` o
JOIN session_info
  ON o.ga_session_id = session_info.ga_session_id
  GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
),


events AS (
SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name,
      COUNT(ep.ga_session_id) AS event_cnt
FROM `DA.event_params` ep
JOIN session_info
ON ep.ga_session_id = session_info.ga_session_id
GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name
),


session AS (
  SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
  FROM session_info
  GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
),


account AS (
  SELECT
  session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
  FROM `data-analytics-mate.DA.account_session` acs
  JOIN session_info
  ON session_info.ga_session_id = acs.ga_session_id
  GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
      )




SELECT
      session_with_orders.date,
      session_with_orders.country,
      session_with_orders.device,
      session_with_orders.continent,
      session_with_orders.channel,
      session_with_orders.test,
      session_with_orders.test_group,
      'session_with_orders' AS event_name,
      session_with_orders.sessions_with_orders AS value
FROM session_with_orders


UNION ALL


SELECT
      events.date,
      events.country,
      events.device,
      events.continent,
      events.channel,
      events.test,
      events.test_group,
      event_name,
      event_cnt AS value
FROM events


UNION ALL


SELECT
      session.date,
      session.country,
      session.device,
      session.continent,
      session.channel,
      session.test,
      session.test_group,
      'session' AS event_name,
      session_cnt AS value
FROM session


UNION ALL


SELECT
      account.date,
      account.country,
      account.device,
      account.continent,
      account.channel,
      account.test,
      account.test_group,
      'new_account' AS event_name,
      new_account_cnt AS value
FROM account
"""

df = client.query(query).to_dataframe(create_bqstorage_client=False)
df.head(11)

In [ ]:
# Швидка перевірка структури даних перед розрахунками
print('Колонки:', list(df.columns))
print('Унікальні тести:', sorted(df['test'].unique()))
print('Унікальні групи:', sorted(df['test_group'].unique()))
print('Приклади event_name:', df['event_name'].unique()[:20])

#4. Визначення метрик

Уся подальша логіка працюватиме циклом по цьому списку: якщо додати сюди п'яту метрику (наприклад, add_to_cart / session), решту коду міняти не потрібно.

In [ ]:
METRICS = [
    {'name': 'add_payment_info / session',  'numerator_event': 'add_payment_info',  'denominator_event': 'session'},
    {'name': 'add_shipping_info / session', 'numerator_event': 'add_shipping_info', 'denominator_event': 'session'},
    {'name': 'begin_checkout / session',    'numerator_event': 'begin_checkout',    'denominator_event': 'session'},
    {'name': 'new_accounts / session',      'numerator_event': 'new_account',       'denominator_event': 'session'},
]

In [ ]:
# Швидка перевірка щоб переконатися, що всі event_name з METRICS і session дійсно є в даних (інакше метрика випаде в подальших розрахунках)
needed_events = {'session'} | {m['numerator_event'] for m in METRICS}
present_events = set(df['event_name'].unique())

print('Відсутні в даних:', needed_events - present_events)

#5. Z-тест для двох пропорцій

Для кожної метрики порівнюємо частку конверсій у групі 1 і групі 2 через z-тест для двох незалежних пропорцій

p1 = x1/n1, p2 = x2/n2 — конверсія в кожній групі
p_pool = (x1+x2)/(n1+n2) — об'єднана (pooled) конверсія
SE = sqrt(p_pool*(1-p_pool)*(1/n1+1/n2)) — стандартна похибка
z = (p1-p2)/SE
p_value — двостороннє значення з нормального розподілу
Якщо p_value < 0.05 (рівень значущості α=5%) → різниця між групами статистично значуща.

In [ ]:
def two_proportion_ztest(x1, n1, x2, n2):
    """
    Z-тест для двох пропорцій.
    x1, x2 — numerator (кількість "успіхів") у групі 1 і 2
    n1, n2 — denominator (розмір вибірки) у групі 1 і 2
    Повертає (z_stat, p_value)
    """
    if n1 == 0 or n2 == 0:
        return np.nan, np.nan

    p1, p2 = x1 / n1, x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))

    if se == 0:
        return np.nan, np.nan

    z = (p1 - p2) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p_value


# Перевірка формули на числах із прикладу-файлу до завдання (test 2, add_payment_info):
z_check, p_check = two_proportion_ztest(2409, 50244, 2344, 50637)
print(f'z = {z_check:.6f}, p = {p_check:.6f}')  # очікується z ≈ 1.240994, p ≈ 0.214608 (НЕ значуще)

#6. Функція розрахунку значущості для довільного зрізу даних

Функція приймає будь-яку підмножину датафрейму (весь `df` — для тоталу, або `df`, відфільтрований по країні/пристрою — для розрізу) і рахує всі 4 метрики для кожного тесту в цьому підмножині. Кількість метрик і кількість тестів **ніде не захардкоджені** — все йде циклами по унікальних значеннях і по списку `METRICS`.

In [ ]:
def calculate_significance(df_subset, metrics, cut_type='total', cut_value='total'):
    results = []

    # 1. Сумуємо value по test / test_group / event_name
    agg = (
        df_subset
        .groupby(['test', 'test_group', 'event_name'], as_index=False)['value']
        .sum()
    )

    # 2. "Розвертаємо" event_name у колонки, щоб зручно брати numerator/denominator
    pivot = agg.pivot_table(
        index=['test', 'test_group'],
        columns='event_name',
        values='value',
        fill_value=0
    ).reset_index()

    for test_number in sorted(pivot['test'].unique()):
        test_slice = pivot[pivot['test'] == test_number].sort_values('test_group')
        groups = test_slice['test_group'].tolist()

        if len(groups) < 2:
            continue  # немає з чим порівнювати (лише 1 група в цьому зрізі)

        group_a, group_b = groups[0], groups[1]
        row_a = test_slice[test_slice['test_group'] == group_a].iloc[0]
        row_b = test_slice[test_slice['test_group'] == group_b].iloc[0]

        # Цикл по METRICS — саме тут немає хардкоду кількості метрик
        for metric in metrics:
            num_event = metric['numerator_event']
            den_event = metric['denominator_event']

            if num_event not in pivot.columns or den_event not in pivot.columns:
                continue

            n1, d1 = row_a[num_event], row_a[den_event]
            n2, d2 = row_b[num_event], row_b[den_event]

            if d1 == 0 or d2 == 0:
                continue

            rate_1 = n1 / d1
            rate_2 = n2 / d2
            z_stat, p_value = two_proportion_ztest(n1, d1, n2, d2)
            metric_change_pct = (rate_2 - rate_1) / rate_1 * 100 if rate_1 else np.nan

            results.append({
                'test_number': test_number,
                'metric': metric['name'],
                'numerator_event': num_event,
                'denominator_event': den_event,
                'cut_type': cut_type,
                'cut_value': cut_value,
                'group_1': group_a,
                'group_2': group_b,
                'numerator_count_1': n1,
                'denominator_count_1': d1,
                'conversion_rate_1': rate_1,
                'numerator_count_2': n2,
                'denominator_count_2': d2,
                'conversion_rate_2': rate_2,
                'metric_change_pct': metric_change_pct,
                'z_stat': z_stat,
                'p_value': p_value,
                'significant': bool(p_value < 0.05) if pd.notna(p_value) else None
            })

    return results

- agg - звичайний groupby().sum(), підсумовує value по всіх датах/країнах/пристроях/каналах, які не входять у зріз (для тоталу, по всіх одразу).
- pivot_table перетворює довгий формат (event_name як значення) на широкий, тепер кожна подія стає окремою колонкою, і можна просто взяти row['add_payment_info'] і row['session'].
- Внутрішній цикл for metric in metrics - саме він робить кількість метрик динамічною.
- cut_type/cut_value це поля, які потім у Tableau дозволять фільтрувати "тотал" окремо від розрізів по країні/пристрою.

#7. Розрахунок в тоталі по кожному тесту

In [ ]:
all_results = []

# Базовий розрахунок в тоталі, без розрізів
all_results += calculate_significance(df, METRICS, cut_type='total', cut_value='total')

print(f'Отримано {len(all_results)} рядків (тести × метрики)')

In [ ]:
pd.DataFrame(all_results)

#8. Розрахунок в розрізах (country / device / continent / channel)

Щоб зробити і показати результат детальніше, рахуємо ту саму значущість окремо для кожного значення в country, device, continent, channel. Список розрізів є масивом: додати новий розріз (наприклад date) можна одним рядком, не змінюючи функцію calculate_significance.

In [ ]:
CUT_DIMENSIONS = ['country', 'device', 'continent', 'channel']

for dim in CUT_DIMENSIONS:
    for value in df[dim].dropna().unique():
        subset = df[df[dim] == value]
        all_results += calculate_significance(subset, METRICS, cut_type=dim, cut_value=value)

print(f'Всього рядків після розрізів: {len(all_results)}')

**Що тут відбувається:** функція calculate_significance з кроку 6 просто викликається ще раз для кожного унікального значення в кожному з 4 вимірів, але вже на відфільтрованому підмножині df. Наприклад, для country == 'Ukraine' вона порахує ті ж 4 метрики × (скільки там тестів трапилось в Україні), з cut_type='country', cut_value='Ukraine'.
Важливий нюанс: якщо в якомусь зрізі (наприклад, рідкісна країна) є дані лише по одній групі тесту, рядок просто не додасться (в функції стоїть if len(groups) < 2: continue), це нормально й очікувано.

In [ ]:
results_df = pd.DataFrame(all_results)

# Скільки рядків не порахувалось коректно (через NaN у p_value)?
print('Рядків з NaN p_value:', results_df['p_value'].isna().sum())

# Прибираємо їх з фінального файлу — вони не несуть статистичної інформації
results_df = results_df.dropna(subset=['p_value'])
print('Залишилось рядків:', len(results_df))

**Чому так вийшло:** формула під коренем p_pool*(1-p_pool) коректна лише коли p_pool в межах 0,1. У SQL-запиті колонка events рахує COUNT(ep.ga_session_id) без DISTINCT, тобто якщо в межах однієї сесії одна й та сама подія, наприклад begin_checkout) спрацювала кілька разів, вона порахується кілька разів. У великому тоталі це непомітно, але в дрібному розрізі. Умовно, одна рідкісна країна з 3 сесіями може вийти, що numerator (кількість подій) перевищує denominator (кількість сесій), тобто "конверсія" виходить більше 100%. Тоді p_pool*(1-p_pool) стає від'ємним, корінь з нього NaN, звідси й попередження.

In [ ]:
# подивись звідки рядки відсіялись
nan_rows = pd.DataFrame(all_results)
nan_rows = nan_rows[nan_rows['p_value'].isna()]
nan_rows['cut_type'].value_counts()

146 із 1956 (~7.5%) це цілком логічно з огляду на кількість країн у датасеті, десятки дрібних країн з малою вибіркою сесій, де й виникає ефект "конверсія >100%" через недублікейтний count.


#9. Формування та збереження фінального CSV

In [ ]:
results_df = results_df.sort_values(['cut_type', 'test_number', 'metric']).reset_index(drop=True)

results_df.to_csv('ab_test_significance_results.csv', index=False)
results_df.head(20)

In [ ]:
from google.colab import files
files.download('ab_test_significance_results.csv')

#10. Висновки

**Результати в тоталі по тестах:**

- **Тест 1** найпомітніший результат: 3 з 4 метрик значущі, і всі три
  з покращенням. `add_payment_info / session` зросла з 4.38% до 4.93%
  (+12.54%, p=0.00009), `add_shipping_info / session`  з 6.69% до 7.13%
  (+6.56%, p=0.0092), `begin_checkout / session`  з 8.34% до 8.90%
  (+6.66%, p=0.0029). Група 2 стабільно конвертує краще по всій верхній
  частині воронки. Лише `new_accounts / session` не показала значущої
  різниці (-3.35%, p=0.123).

- **Тест 2** жодної значущої різниці за жодною з 4 метрик (p-value від
  0.21 до 0.56). Групи статистично не відрізняються.

- **Тест 3**  значущий лише `begin_checkout / session`: 13.61% → 13.15%
  (-3.35%, p=0.012), тобто зниження. Інші три метрики в межах статистичного
  шуму.

- **Тест 4**  2 з 4 метрик значущі, обидві зі зниженням: `begin_checkout /
  session` (11.95% → 11.67%, -2.35%, p=0.046) і `new_accounts / session`
  (8.55% → 8.26%, -3.36%, p=0.018). `add_payment_info` і `add_shipping_info`
  не значущі.

**Розрізи (country / device / continent / channel):**

- Загалом статистично значущими виявились **20.8% усіх перевірених
  комбінацій** (376 з 1810, з урахуванням розрізів).
- Розрізи по `channel` (40.0% значущих) і `device` (35.4%) дають вищу частку
  значущих результатів, ніж `country` (18.7%) - логічно, бо по країнах
  вибірки часто набагато дрібніші й статистична потужність нижча.
- Найчутливіша до змін метрика - **`begin_checkout / session`**: значуща
  в тоталі у 3 з 4 тестів і найчастіше значуща з усіх метрик у розрізах
  (170 з 450, 37.8%). Найстабільніша й найменш чутлива - **`new_accounts /
  session`** (лише 23 з 485, 4.7% значущих результатів по всіх зрізах) -
  реєстрація акаунту, ймовірно, слабо пов'язана з тим, що саме змінювалося
  в тестах.

**Загальний висновок:**

Найпереконливіший тест - **тест 1**: зміни в групі 2 дали значуще покращення
одразу по трьох верхньоворонкових метриках, тому саме цей варіант варто
розглядати для розкочування на всіх користувачів. Тест 2 ефекту не показав.
Тести 3 і 4 навпаки показали значуще **зниження** конверсії в групі 2 по
`begin_checkout` (і в тесті 4 ще й по `new_accounts`) - тобто варіанти цих
тестів радше гірші за контроль і розкочувати їх не варто.

**Технічне зауваження:** 146 з 1956 розрахованих комбінацій (усі - в розрізі
`country`) довелось виключити з фінального файлу через некоректний z-тест:
вихідний SQL рахує події без `DISTINCT`, тому в дрібних країнах подія могла
траплятись частіше за кількість сесій, формально даючи конверсію понад 100%.

**Посилання:**
- Дашборд у Tableau:
- CSV з результатами розрахунків: *[https://drive.google.com/file/d/1y9fDjQoTUaI-d7s9iMnFXy1CdJtB_QBr/view?usp=sharing]*